# 11 — Baseline Evaluation (Zero-Shot, Few-Shot, CoT)
Runs four prompt strategies on GSM8K 500-subset, MATH 200-subset, and 120 WSU domain cases.
Results saved to `data/results/baseline_results.json`.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))

from evaluation.benchmarks import BenchmarkLoader
from evaluation.metrics import EvaluationMetrics
from llm.cache import ResponseCache
from prompts.template import PromptTemplate
from prompts.mutations import PromptMutator

cache = ResponseCache(db_path=str(REPO_ROOT / "data" / "cache" / "responses.db"))
loader = BenchmarkLoader(cache_dir=REPO_ROOT / "data" / "benchmarks")
results_path = REPO_ROOT / "data" / "results" / "baseline_results.json"
results_path.parent.mkdir(parents=True, exist_ok=True)

## Load Datasets

In [ ]:
gsm8k = loader.load_gsm8k(subset_size=500)
math  = loader.load_math(subset_size=200)

# WSU domain cases — load from benchmarks cache if available
wsu_path = REPO_ROOT / "data" / "benchmarks" / "test_cases.json"
wsu = json.loads(wsu_path.read_text())[:120] if wsu_path.exists() else []

print(f"GSM8K : {len(gsm8k)} examples")
print(f"MATH  : {len(math)} examples")
print(f"WSU   : {len(wsu)} examples")

## Define Prompt Strategies

In [ ]:
def make_zero_shot(question):
    return f"Answer the following question.\n\n{question}"

def make_few_shot(question, examples):
    ex_block = "\n\n".join(f"Q: {e['question']}\nA: {e['answer']}" for e in examples[:3])
    return f"{ex_block}\n\nQ: {question}\nA:"

def make_cot(question):
    return f"Answer the following question. Let us think step by step.\n\n{question}"

def make_self_consistency(question):
    return make_cot(question)  # same prompt, called 5x with temperature > 0

def majority_vote(answers):
    from collections import Counter
    return Counter(answers).most_common(1)[0][0]

## Evaluation Runner
> **Note:** Replace `mock_llm_call` with your actual LLM client to run live. Cached responses are reused automatically.

In [ ]:
def mock_llm_call(prompt, temperature=0.0):
    """Stub — replace with real LLM client call."""
    cached = cache.get(prompt, model="claude-3-haiku", temperature=temperature)
    if cached:
        return cached, {"input_tokens": 0, "output_tokens": 0}
    response = "[mock response]"
    cache.set(prompt, model="claude-3-haiku", temperature=temperature, response=response)
    return response, {"input_tokens": len(prompt.split()), "output_tokens": 5}

def evaluate_strategy(dataset, strategy_fn, sc_samples=1):
    predictions, total_usage = [], {"input_tokens": 0, "output_tokens": 0}
    for ex in dataset:
        prompt = strategy_fn(ex["question"])
        if sc_samples > 1:
            answers = []
            for _ in range(sc_samples):
                resp, usage = mock_llm_call(prompt, temperature=0.7)
                answers.append(resp)
                total_usage["input_tokens"]  += usage["input_tokens"]
                total_usage["output_tokens"] += usage["output_tokens"]
            predictions.append(majority_vote(answers))
        else:
            resp, usage = mock_llm_call(prompt)
            predictions.append(resp)
            total_usage["input_tokens"]  += usage["input_tokens"]
            total_usage["output_tokens"] += usage["output_tokens"]
    ground_truth = [ex["answer"] for ex in dataset]
    acc = EvaluationMetrics.accuracy(predictions, ground_truth)
    cost = EvaluationMetrics.token_cost(total_usage)
    avg_tokens = (total_usage["input_tokens"] + total_usage["output_tokens"]) / max(len(dataset), 1)
    return acc, avg_tokens, cost

## Run Baselines

In [ ]:
strategies = [
    ("zero_shot",        lambda q: make_zero_shot(q),          1),
    ("few_shot",         lambda q: make_few_shot(q, gsm8k[:3]), 1),
    ("cot",              lambda q: make_cot(q),                 1),
    ("self_consistency", lambda q: make_self_consistency(q),    5),
]

datasets = [
    ("gsm8k", gsm8k),
    ("math",  math),
]
if wsu:
    datasets.append(("wsu", wsu))

rows = []
for ds_name, ds in datasets:
    for method, fn, sc in strategies:
        acc, avg_tok, cost = evaluate_strategy(ds, fn, sc_samples=sc)
        row = {"dataset": ds_name, "method": method,
               "accuracy": round(acc, 4), "avg_tokens": round(avg_tok, 1),
               "estimated_cost": round(cost, 6)}
        rows.append(row)
        print(row)

## Save Results

In [ ]:
results_path.write_text(json.dumps(rows, indent=2))
print(f"Saved {len(rows)} rows to {results_path}")